# Analisis Eksperimen Robot Path Planning

Notebook ini mendampingi makalah ASA tentang perbandingan **Brute Force**, **UCS**, **GBFS**, **A\***, dan **RRT\***. Implementasi algoritma tetap berada di package `asa_path_planning`; notebook berfokus pada reproduksi, pemeriksaan data, visualisasi, dan interpretasi hasil.

## 1. Persiapan

Jalankan notebook dari root project dengan `uv run --extra notebook jupyter lab notebooks/analisis_path_planning.ipynb`. Kode berikut mencari root project secara otomatis agar notebook juga tetap bekerja ketika kernel memulai direktori kerja dari folder `notebooks/`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from asa_path_planning.experiment import (
    ALGORITHM_ORDER,
    SCENARIOS,
    run_experiment,
    scenario_from_seed,
)


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Root project dengan pyproject.toml tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "results" / "data"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

print(f"Project root: {PROJECT_ROOT}")
print(f"Skenario: {', '.join(SCENARIOS)}")
print(f"Algoritma: {', '.join(ALGORITHM_ORDER)}")

ModuleNotFoundError: No module named 'pandas'

## 2. Memuat ringkasan eksperimen

`ringkasan_eksperimen.csv` berisi rata-rata dan simpangan baku dari lima seed untuk setiap pasangan skenario-algoritma. Metrik `avg_expanded` adalah jumlah `processed`: kandidat rute untuk Brute Force, simpul grid untuk UCS/GBFS/A\*, dan node sampling untuk RRT\*. Karena definisinya berbeda, perbandingan langsung paling adil dilakukan di dalam keluarga representasi yang sama.

In [ ]:
summary = pd.read_csv(DATA_DIR / "ringkasan_eksperimen.csv")
summary["scenario"] = pd.Categorical(summary["scenario"], SCENARIOS, ordered=True)
summary["algorithm"] = pd.Categorical(
    summary["algorithm"], ALGORITHM_ORDER, ordered=True
)
summary = summary.sort_values(["scenario", "algorithm"]).reset_index(drop=True)

display(summary.round(2))

## 3. Perbandingan processed pada algoritma grid

UCS, GBFS, dan A\* sama-sama memproses simpul pada grid 60 x 40, sehingga nilai processed ketiganya dapat dibandingkan langsung. Tabel pivot berikut adalah sumber data Figure 3 pada makalah.

In [ ]:
grid_algorithms = ["UCS", "GBFS", "A*"]
processed_grid = (
    summary[summary["algorithm"].isin(grid_algorithms)]
    .pivot(index="scenario", columns="algorithm", values="avg_expanded")
    .reindex(index=SCENARIOS, columns=grid_algorithms)
)

display(processed_grid.round(1))
display(Image(filename=str(FIGURES_DIR / "fig_processed_grid.png")))

## 4. Verifikasi klaim A\* dibandingkan UCS

A\* dan UCS menghasilkan biaya serta tingkat keberhasilan yang sama pada setiap skenario. Perbedaannya terletak pada banyaknya simpul yang perlu diproses. Sel berikut menghitung persentase pengurangan processed A\* terhadap UCS langsung dari CSV.

In [ ]:
ucs = summary[summary["algorithm"] == "UCS"].set_index("scenario")
astar = summary[summary["algorithm"] == "A*"].set_index("scenario")

comparison = pd.DataFrame(
    {
        "UCS processed": ucs["avg_expanded"],
        "A* processed": astar["avg_expanded"],
        "pengurangan processed (%)": (
            (ucs["avg_expanded"] - astar["avg_expanded"])
            / ucs["avg_expanded"]
            * 100
        ),
        "biaya sama": ucs["avg_cost"].round(10) == astar["avg_cost"].round(10),
        "keberhasilan sama": ucs["success_rate"] == astar["success_rate"],
    }
)

display(comparison.round(1))
assert comparison["biaya sama"].all()
assert comparison["keberhasilan sama"].all()

Hasilnya memverifikasi rentang pengurangan processed **42,6-87,9%**. A\* memperoleh manfaat terbesar pada skenario Mudah dan Sedang, ketika jarak Euclidean cukup informatif untuk mengarahkan frontier menuju goal.

## 5. Mengapa processed A\* meningkat pada skenario Sulit?

Pada skenario Sulit, A\* memproses **1.037 simpul**, atau sekitar **57,4%** dari processed UCS. Rasio ini jauh lebih tinggi daripada Mudah (12,1%) dan Sedang (12,4%). Konfigurasi dua dinding dan delapan hambatan memaksa lintasan melakukan detour melalui ruang bebas terbatas. Akibatnya, jarak Euclidean ke goal kurang mewakili biaya sisa yang sebenarnya; banyak simpul tampak menjanjikan secara heuristik tetapi tetap harus diperiksa sebelum jalur optimal ditemukan.

Interpretasi ini menjelaskan pola eksperimen, bukan membuktikan hubungan kausal secara terpisah. Pembuktian yang lebih kuat dapat dilakukan dengan mencatat urutan ekspansi frontier dan membandingkan distribusi nilai `g(n)` serta `h(n)`.

In [ ]:
processed_ratio = (
    processed_grid["A*"] / processed_grid["UCS"] * 100
).rename("A* processed relatif terhadap UCS (%)")

display(processed_ratio.to_frame().round(1))

axis = processed_ratio.plot(
    kind="bar",
    color="#2ca02c",
    figsize=(7, 3.5),
    ylim=(0, 65),
    legend=False,
)
axis.set_xlabel("Skenario")
axis.set_ylabel("A* / UCS processed (%)")
axis.set_title("Kedekatan jumlah processed A* terhadap UCS")
axis.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Memeriksa konfigurasi skenario Sulit

Peta dibangkitkan secara deterministik dari nama skenario dan seed. Cell ini menampilkan jumlah hambatan pada seed yang dipakai Figure 1, tanpa menduplikasi logika pembangkitan peta.

In [ ]:
detail_seed = 4
sulit = scenario_from_seed("Sulit", 1000 * len("Sulit") + detail_seed)

print(f"Ukuran grid: {sulit.width} x {sulit.height}")
print(f"Start: {sulit.start}; goal: {sulit.goal}")
print(f"Hambatan lingkaran: {len(sulit.circles)}")
print(f"Hambatan persegi panjang/dinding: {len(sulit.rects)}")
display(Image(filename=str(FIGURES_DIR / "fig_perbandingan_jalur.png")))

## 7. Menjalankan ulang eksperimen

Aktifkan `RUN_FULL_EXPERIMENT` untuk menjalankan seluruh 4 skenario x 5 seed x 5 algoritma. Output notebook ditulis ke `notebooks/output/`, sehingga hasil utama di `results/` tidak tertimpa. Biaya, keberhasilan, dan processed bersifat deterministik untuk seed yang sama, sedangkan waktu eksekusi dapat berubah mengikuti beban perangkat.

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    notebook_output = PROJECT_ROOT / "notebooks" / "output"
    artifacts = run_experiment(
        data_dir=notebook_output / "data",
        figures_dir=notebook_output / "figures",
    )
    print("Eksperimen selesai.")
    for artifact in artifacts[:-1]:
        print(artifact)
else:
    print("Set RUN_FULL_EXPERIMENT = True untuk menjalankan eksperimen penuh.")

## Ringkasan

- A\* mempertahankan biaya dan tingkat keberhasilan UCS pada seluruh skenario.
- Pengurangan processed A\* terhadap UCS berada pada rentang 42,6-87,9%.
- Skenario Sulit adalah pengecualian penting: heuristik Euclidean menjadi kurang informatif karena lintasan perlu melakukan detour di antara hambatan.
- Runtime perlu dibaca bersama simpangan baku karena dipengaruhi kondisi eksekusi perangkat.